#Task 1 Code Cell: Load and Inspect the Data

In [1]:
#Import required libraries for data handling
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#Load movies and ratings datasets into pandas DataFrames
movies_df = pd.read_csv('movies.csv')
ratings_df = pd.read_csv('ratings.csv')

#Display the first five rows of each dataset to verify structure
print("--- Movies Dataset (First 5 Rows) ---")
display(movies_df.head())

print("\n--- Ratings Dataset (First 5 Rows) ---")
display(ratings_df.head())

#Display the number of rows and columns in each dataset
print(f"\nMovies Shape: {movies_df.shape[0]} rows, {movies_df.shape[1]} columns")
print(f"Ratings Shape: {ratings_df.shape[0]} rows, {ratings_df.shape[1]} columns")

#Check for missing values in both datasets
print("\n--- Missing Values Check ---")
print("Movies missing values:\n", movies_df.isnull().sum())
print("Ratings missing values:\n", ratings_df.isnull().sum())

--- Movies Dataset (First 5 Rows) ---


,movie_id,title,genres
0,1,Inception,Sci-Fi|Thriller
1,2,Interstellar,Sci-Fi|Drama
2,3,The Matrix,Sci-Fi|Action
3,4,Arrival,Sci-Fi|Drama
4,5,Edge of Tomorrow,Sci-Fi|Action



--- Ratings Dataset (First 5 Rows) ---


,user_id,movie_id,rating
0,1,2,3.5
1,1,3,4.5
2,1,5,3.5
3,1,6,3.5
4,1,11,3.0



Movies Shape: 36 rows, 3 columns
Ratings Shape: 440 rows, 3 columns

--- Missing Values Check ---
Movies missing values:
 movie_id    0
title       0
genres      0
dtype: int64
Ratings missing values:
 user_id     0
movie_id    0
rating      0
dtype: int64


Upon inspecting movies.csv and ratings.csv, isnull().sum() confirmed that zero missing values exist across all columns in both datasets. The movie catalogue contains clean titles, genres, and IDs, while the ratings file has complete user-item interaction records, allowing us to proceed directly without data cleaning.

#Task 2 – Profile Your Assigned User

In [2]:
#Profile Assigned User 17
ASSIGNED_USER_ID = 17

#Filter ratings dataset for the assigned user
user_ratings = ratings_df[ratings_df['user_id'] == ASSIGNED_USER_ID]

#Merge filtered ratings with the movie catalogue to include titles and genres
user_profile = user_ratings.merge(movies_df, on='movie_id')

#Display all movies rated by the assigned user
print(f"--- All Movies Rated by User {ASSIGNED_USER_ID} ---")
display(user_profile[['title', 'genres', 'rating']])

#Identify and display the user's three highest-rated movies
top_3_rated = user_profile.sort_values(by='rating', ascending=False).head(3)
print(f"\n--- Top 3 Highest-Rated Movies by User {ASSIGNED_USER_ID} ---")
display(top_3_rated[['title', 'genres', 'rating']])

--- All Movies Rated by User 17 ---


,title,genres,rating
0,The Matrix,Sci-Fi|Action,4.0
1,Arrival,Sci-Fi|Drama,3.5
2,Blade Runner 2049,Sci-Fi|Drama,4.0
3,The Dark Knight,Action|Crime,4.0
4,John Wick,Action|Thriller,3.5
5,Mad Max: Fury Road,Action|Adventure,3.0
6,Mission: Impossible - Fallout,Action|Thriller,4.0
7,The Shawshank Redemption,Drama,3.0
8,Toy Story,Animation|Comedy,2.0
9,A Quiet Place,Horror|Thriller,4.0



--- Top 3 Highest-Rated Movies by User 17 ---


,title,genres,rating
0,The Matrix,Sci-Fi|Action,4.0
2,Blade Runner 2049,Sci-Fi|Drama,4.0
3,The Dark Knight,Action|Crime,4.0


User 17 has rated 11 movies in total, showing a preference for Sci-Fi, Action, and Thriller genres. Their highest-rated films (4.0/5.0) include The Matrix, Blade Runner 2049, The Dark Knight, Mission: Impossible - Fallout, and A Quiet Place. This demonstrates that User 17 prefers genre-driven, high-concept storytelling and action-oriented cinema rather than light comedy or standard romance.

#Task 3 – Build the Recommendation Logic

In [3]:
#1: Identify positively rated movies (rating >= 4.0)
liked_movies = user_profile[user_profile['rating'] >= 4.0]

#2: Convert genres into numerical features using CountVectorizer (handling pipe separator)
cv = CountVectorizer(token_pattern=r'[^|]+')
genre_matrix = cv.fit_transform(movies_df['genres'])

#3: Calculate cosine similarity between all movies based on genre features
cosine_sim = cosine_similarity(genre_matrix, genre_matrix)

#Map liked movies to their corresponding indices in the movies DataFrame
liked_indices = movies_df[movies_df['movie_id'].isin(liked_movies['movie_id'])].index.tolist()

#Aggregate similarity scores across all movies the user liked
sim_scores = np.zeros(len(movies_df))
for idx in liked_indices:
    sim_scores += cosine_sim[idx]

movies_df['similarity_score'] = sim_scores

#4: Exclude already rated movies and recommend top 5 unrated movies
rated_movie_ids = user_ratings['movie_id'].tolist()
unrated_movies = movies_df[~movies_df['movie_id'].isin(rated_movie_ids)].copy()

recommendations = unrated_movies.sort_values(by='similarity_score', ascending=False).head(5)

print("--- Top 5 Generated Recommendations ---")
display(recommendations[['movie_id', 'title', 'genres', 'similarity_score']])

--- Top 5 Generated Recommendations ---


,movie_id,title,genres,similarity_score
4,5,Edge of Tomorrow,Sci-Fi|Action,2.5
0,1,Inception,Sci-Fi|Thriller,2.0
1,2,Interstellar,Sci-Fi|Drama,1.5
9,10,Black Panther,Action|Superhero,1.5
10,11,Avengers: Endgame,Action|Superhero,1.5


#Task 4 – Explain the Recommendations

In [4]:
#Create Clean Recommendation Results Table with Reasons

rec_data = [
    {
        "Rank": 1,
        "Recommended Movie": "Edge of Tomorrow",
        "Genres": "Sci-Fi|Action",
        "Reason": "Shares the core 'Sci-Fi|Action' genres with highly-rated favourites like The Matrix and matches user preference for fast-paced action."
    },
    {
        "Rank": 2,
        "Recommended Movie": "Inception",
        "Genres": "Sci-Fi|Thriller",
        "Reason": "Aligns strongly with User 17's high ratings for complex Sci-Fi and Thriller titles such as Blade Runner 2049 and A Quiet Place."
    },
    {
        "Rank": 3,
        "Recommended Movie": "Interstellar",
        "Genres": "Sci-Fi|Drama",
        "Reason": "Connects directly to the user's appreciation for immersive Sci-Fi narratives seen in Arrival and Blade Runner 2049."
    },
    {
        "Rank": 4,
        "Recommended Movie": "Black Panther",
        "Genres": "Action|Superhero",
        "Reason": "Matches the user's strong affinity for 'Action' movies, echoing elements from The Dark Knight and Mission: Impossible."
    },
    {
        "Rank": 5,
        "Recommended Movie": "Avengers: Endgame",
        "Genres": "Action|Superhero",
        "Reason": "Capitalizes on the user's proven engagement with high-stakes 'Action' features like The Dark Knight and John Wick."
    }
]

results_df = pd.DataFrame(rec_data)
print("--- Final Personalized Movie Recommendations ---")
display(results_df)

--- Final Personalized Movie Recommendations ---


,Rank,Recommended Movie,Genres,Reason
0,1,Edge of Tomorrow,Sci-Fi|Action,Shares the core 'Sci-Fi|Action' genres with hi...
1,2,Inception,Sci-Fi|Thriller,Aligns strongly with User 17's high ratings fo...
2,3,Interstellar,Sci-Fi|Drama,Connects directly to the user's appreciation f...
3,4,Black Panther,Action|Superhero,Matches the user's strong affinity for 'Action...
4,5,Avengers: Endgame,Action|Superhero,Capitalizes on the user's proven engagement wi...


#Task 5 – Critical Thinking

##Limitation:
The content-based recommender relies entirely on categorical genre tags using CountVectorizer, ignoring richer semantic features like director, cast members, plot synopses, and user rating magnitudes (treating a 4.0 and 5.0 rating identically).

##Suggested Improvement:
Incorporate TF-IDF vectorization on movie plot descriptions combined with user rating weights (giving more emphasis to 5.0-rated films than 4.0-rated ones) to capture deeper theme similarities and nuance.